In [0]:
!pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

In [0]:
import os
import sys
from pyspark.sql import SparkSession

# 3. Azure Key (Ensure your .env is loaded or export AZURE_STORAGE_KEY first)
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

# Set the key on the DFS endpoint
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

print(f"🚀 Spark {spark.version} Ready for ADLS Gen2!")

In [0]:
# --- 2. PATH DEFINITIONS ---
# Updated to match your folder structure in Screenshot 3:23:46 PM
bronze_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

silver_quotes_path = f"{silver_base_path}/crypto_quotes"
silver_global_path = f"{silver_base_path}/global_metrics"
silver_historical_path = f"{silver_base_path}/historical_prices"

# --- 3. LOAD BRONZE DATA ---
# We read as JSON and use recursive lookup to grab files from both cmc and coingecko subfolders
bronze_df = spark.read.option("recursiveFileLookup", "true") \
    .json(bronze_base_path)

# Creating temporary aliases to match your transformation logic names
# Your ingestion script wraps everything in 'payload', 'ingested_at', 'source', etc.
bronze_df = bronze_df.withColumnRenamed("ingested_at", "event_timestamp")

print(f"✅ Bronze JSON data loaded. Total rows: {bronze_df.count()}")

In [0]:
from pyspark.sql.functions import col, from_json, explode, get_json_object, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

# --- A. PROCESS REAL-TIME QUOTES (From Listings Endpoint) ---
# Note: payload is an array of coins
cmc_listings_schema = ArrayType(StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("symbol", StringType()),
    StructField("quote", StructType([
        StructField("USD", StructType([
            StructField("price", DoubleType()),
            StructField("volume_24h", DoubleType()),
            StructField("market_cap", DoubleType())
        ]))
    ]))
]))

silver_quotes = bronze_df.filter(col("endpoint_type") == "listings") \
    .withColumn("parsed_payload", from_json(col("payload"), cmc_listings_schema)) \
    .withColumn("coin", explode(col("parsed_payload"))) \
    .select(
        col("coin.id").alias("coin_id"),
        col("coin.symbol"),
        col("coin.name"),
        col("coin.quote.USD.price").alias("price_usd"),
        col("coin.quote.USD.volume_24h").alias("volume_24h"),
        col("coin.quote.USD.market_cap").alias("market_cap"),
        col("event_timestamp"),
        current_timestamp().alias("silver_processing_timestamp")
    )

silver_global = bronze_df.filter(col("endpoint_type") == "global") \
    .select(
        get_json_object(col("payload"), "$.quote.USD.total_market_cap").cast("double").alias("total_market_cap"),
        get_json_object(col("payload"), "$.quote.USD.total_volume_24h").cast("double").alias("total_volume_24h"),
        get_json_object(col("payload"), "$.btc_dominance").cast("double").alias("btc_dominance"),
        col("event_timestamp")
    )

In [0]:
# --- C. PROCESS HISTORICAL BACKFILL ---
# Schema for CoinGecko's nested arrays: [[timestamp, value], ...]
coingecko_schema = StructType([
    StructField("prices", ArrayType(ArrayType(DoubleType()))),
    StructField("market_caps", ArrayType(ArrayType(DoubleType()))),
    StructField("total_volumes", ArrayType(ArrayType(DoubleType())))
])

# 1. Select the raw rows for historical data
historical_df = bronze_df.filter(col("endpoint_type") == "historical_backfill")

# 2. Parse payload and explode prices
# Note: Using col("payload") directly instead of raw_content
silver_historical = historical_df.withColumn("parsed", from_json(col("payload"), coingecko_schema)) \
    .withColumn("exploded_prices", explode(col("parsed.prices"))) \
    .select(
        col("coin_id"), # Injected by your ingestion script
        (col("exploded_prices")[0] / 1000).cast("timestamp").alias("price_timestamp"),
        col("exploded_prices")[1].alias("price_usd"),
        col("source"),
        current_timestamp().alias("silver_processing_timestamp")
    )

In [0]:
# --- D. SAVE AS DELTA TABLES ---
print("Writing Silver tables to Azure...")

# Define write function to keep it clean
def save_to_silver(df, path):
    df.write.format("delta") \
      .mode("overwrite") \
      .option("mergeSchema", "true") \
      .save(path)

save_to_silver(silver_quotes, silver_quotes_path)
save_to_silver(silver_global, silver_global_path)
save_to_silver(silver_historical, silver_historical_path)

print("✅ Silver Layer Processing Complete!")

In [0]:
# --- E. VERIFY SAMPLES ---
print("Silver Quotes Sample:")
spark.read.format("delta").load(silver_quotes_path).show(5)

print("Silver Historical Sample:")
spark.read.format("delta").load(silver_historical_path).show(5)

print("Silver Global Metrics Sample:")
spark.read.format("delta").load(silver_global_path).show(5)